# Sélection de caractéristiques

Comme nous l'avons vu en cours, les approches de machine learning ne sont pas réellement capables de
sélectionner *seules* les bonnes caractéristiques: il est donc essentiel de leur fournir l'information utile
et d'éliminer (au moins une part de) l'information inutile.

Ce notebook reproduit l'essentiel des expériences vues dans les transparents de cours: ce sont les exemples
que nous trouvons les plus parlants pour mettre en évidence l'importance de cette tâche critique.

**Progression du notebook**: on commence par *constater le problème* — ajouter des variables inutiles dégrade
réellement les performances, c'est le **fléau de la dimensionnalité** (A). On y répond ensuite par trois
familles de solutions, présentées dans l'ordre où elles interviennent dans une chaine de traitement:

* les méthodes **supervisées** qui choisissent un sous-ensemble des variables existantes, soit en les notant
  une par une (*filtre*, B.1), soit en évaluant des sous-ensembles avec un modèle (*enveloppe*, B.2);
* la réduction de dimension **non supervisée** par ACP, qui ne sélectionne pas mais *construit* de nouvelles
  variables (C);
* la **régularisation**, qui fait la sélection pendant l'apprentissage lui-même (D).

La partie E propose des exercices complémentaires indépendants.

**Le vocabulaire de la littérature**, à bien distinguer:

| Famille | Principe | Coût de calcul | Où dans ce notebook |
|---|---|---|---|
| **Filtre** (*filter*) | chaque variable est notée indépendamment des autres et du modèle | très faible | B.1, E.1 |
| **Enveloppe** (*wrapper*) | des sous-ensembles sont essayés et évalués *pour un modèle donné* | élevé | B.2, E.5 |
| **Intégrée** (*embedded*) | la sélection est un effet de bord de l'apprentissage | faible | D, E.6 |
| **Extraction** (*feature extraction*) | on ne choisit pas, on recombine les variables | faible | C, E.4 |

**Conventions**: les cellules à compléter sont signalées par un commentaire `TODO` dans la version étudiante ;
les blocs <span style="color:red">Mini-exo</span> sont de courts exercices d'application immédiate.

<a id="sec-plan"></a>
## Plan

* [A. Le fléau de la dimensionnalité](#sec-a)
    * [A.1 Génération des données jouets](#sec-a1)
    * [A.2 Un problème facile... en 2 dimensions](#sec-a2)
    * [A.3 Mise en évidence du fléau](#sec-a3)
* [B. Sélectionner un sous-ensemble de variables](#sec-b)
    * [B.1 Filtre naïf: corrélation entre chaque variable et $y$](#sec-b1)
    * [B.2 Sélection séquentielle: une approche enveloppe](#sec-b2)
* [C. Construire de nouvelles variables: l'ACP](#sec-c)
    * [C.1 Valeurs propres: combien d'axes conserver?](#sec-c1)
    * [C.2 Performances en fonction du nombre d'axes](#sec-c2)
    * [C.3 Réduction en 2D: la visualisation](#sec-c3)
    * [C.4 Application aux données USPS](#sec-c4)
* [D. Régularisation: sélectionner pendant l'apprentissage](#sec-d)
    * [D.1 Régularisation L2 (Ridge)](#sec-d1)
    * [D.2 Régularisation L1 (LASSO)](#sec-d2)
    * [D.3 Régularisation Elastic Net](#sec-d3)
    * [D.4 Synthèse des trois régularisations](#sec-d4)
* [E. Exercices complémentaires](#sec-e)
    * [E.1 Filtres univariés et leur angle mort](#sec-e1)
    * [E.2 Variables redondantes: filtre contre enveloppe](#sec-e2)
    * [E.3 Sélectionner avant la validation croisée: une fuite de données](#sec-e3)
    * [E.4 Deux pièges de l'ACP: l'échelle et l'absence d'étiquettes](#sec-e4)
    * [E.5 Élimination récursive (RFE) et choix automatique du nombre de variables](#sec-e5)
    * [E.6 Stabilité de la sélection: LASSO contre Elastic Net](#sec-e6)
* [Annexe: transformation du notebook en version étudiante](#sec-annexe)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn import svm
from sklearn.metrics import accuracy_score
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split

%matplotlib inline

<a id="sec-a"></a>
## A. Le fléau de la dimensionnalité

*Curse of dimensionality*

L'intuition trompeuse est la suivante: *une variable de plus, c'est une information de plus, donc ça ne peut
pas faire de mal*. C'est faux. En grande dimension, les points deviennent tous à peu près équidistants les uns
des autres: un modèle qui raisonne sur des distances (plus proches voisins, SVM à noyau gaussien, ...) perd
alors toute capacité à généraliser, et se contente d'apprendre par coeur les données d'apprentissage.

Le protocole de cette partie est volontairement caricatural: on part d'un problème à 2 dimensions **facile**,
et on lui ajoute des dimensions de **bruit pur** — qui ne contiennent, par construction, strictement aucune
information sur les étiquettes. Toute dégradation observée sera donc entièrement imputable à la dimension.

<a id="sec-a1"></a>
### A.1 Génération des données jouets

Deux gaussiennes bien séparées en 2 dimensions, auxquelles on concatène 20 dimensions de bruit.

**Note**: le `np.random.seed(0)` rend le bruit reproductible d'une exécution à l'autre. Sans lui, les
courbes changent à chaque `Restart & Run All`, ce qui rend toute discussion des résultats impossible.

In [ ]:
np.random.seed(0)   # reproductibilité du bruit tiré ci-dessous

# 1. le signal: 100 points, 2 classes, 2 dimensions
centers      = [[-2.0, -2.0], [2.0, 2.0]]
clusters_std = [1.5, 1.5]
X, y = make_blobs(n_samples=100, centers=centers, cluster_std=clusters_std,
                  n_features=2, random_state=0)

# 2. le bruit: 20 dimensions supplémentaires SANS aucune information sur y
#    (même écart-type que le signal pour qu'elles soient indiscernables à l'oeil)
ndim_noise = 20
Noise      = np.random.randn(len(X), ndim_noise) * clusters_std[0]
Xn         = np.concatenate((X, Noise), axis=1)

# 3. séparation apprentissage / test
X_train, X_test, y_train, y_test = train_test_split(Xn, y, test_size=0.33, random_state=0)

print("2 dimensions utiles + {} dimensions de bruit = {} dimensions".format(ndim_noise, Xn.shape[1]))
print("{} exemples d'apprentissage, {} de test".format(len(X_train), len(X_test)))

In [ ]:
# visualisation de la matrice de données complète: une ligne = un exemple, une colonne = une variable
# les exemples sont triés par classe pour faire ressortir la structure

ind = np.argsort(y_train)

plt.figure(facecolor='white', figsize=[6,5])
plt.imshow(X_train[ind], aspect='auto')
plt.colorbar(label="valeur")
plt.xlabel("variables (les 2 premières sont les seules utiles)")
plt.ylabel("exemples (triés par classe)")
plt.axvline(1.5, color='r', lw=2)                 # frontière signal / bruit
plt.title("Saurez-vous retrouver l'information à l'oeil?")
#plt.savefig("fig/Xbruit.png", bbox_inches='tight')

Seules les deux colonnes de gauche (à gauche du trait rouge) séparent les exemples du haut de ceux du bas:
le dégradé y suit le tri par classe. Les 20 autres colonnes sont un damier sans structure. C'est très visible
ici parce que nous savons où regarder — sur des données réelles, personne ne connait la position du trait
rouge, et c'est précisément tout le problème.

<a id="sec-a2"></a>
### A.2 Un problème facile... en 2 dimensions

1. Vérifier la dimension des données générées ci-dessus
2. Afficher le nuage de points en utilisant les deux premières dimensions de `X_train`
3. Apprendre un SVM à noyau gaussien sur ces deux seules dimensions et mesurer sa performance en test:
   c'est la **référence** à laquelle nous comparerons tout le reste du notebook

In [ ]:
# Dimension de X

# Scatter plot de X (deux premières dimensions)

# Performance de référence sur les 2 dimensions utiles

# 4 lignes attendues:
#  1. print des .shape de X_train et X_test (combien de colonnes? combien sont réellement utiles?)
#  2. plt.scatter([...], c=y_train) + labels des axes
#  3. mod_ref = svm.SVC(gamma=0.1) appris sur X_train... Mais SEULEMENT sur les 2 colonnes utiles
#  4. perf_ref = accuracy_score([...]) attention à être cohérent sur les dimensions de X
#     => à garder en tête: c'est la référence de tout le reste du notebook

###  TODO  ###

<a id="sec-a3"></a>
### A.3 Mise en évidence du fléau

Reproduisons maintenant l'expérience du cours: on repart des 2 dimensions utiles et on rajoute les dimensions
de bruit **progressivement**, en mesurant à chaque étape la performance en apprentissage et en test.

<img src = "fig/curse.png">

À compléter dans la cellule ci-dessous:
1. extraire les `ndim` premières colonnes des données
2. apprendre le modèle
3. évaluer en apprentissage et en test, et stocker les résultats pour le tracé

In [ ]:
# reproduction de l'expérience sur le fléau de la dimensionalité

# gamma est FIXE pendant toute l'expérience: c'est volontaire, cela revient à supposer une échelle de
# distance constante alors que la distance moyenne entre deux points, elle, croit avec la dimension.
# C'est exactement le mécanisme qui met le sur-apprentissage en évidence.
mod = svm.SVC(gamma=0.1)

perf_train = []
perf_test  = []
all_ndim   = list(range(2, X_train.shape[1] + 1, 2))   # 2, 4, 6, ... 22 dimensions

for ndim in all_ndim:
    # 1. extraction des données temporaires
    #    xtmp = X_train[...]  (sélectionner les ndim premières colonnes = 2 utiles + du bruit)
    # 2. apprentissage du modèle: mod.fit(xtmp, y_train)   -- mod est défini une fois AVANT la boucle
    # 3. évaluation et stockage (pour le tracé de la courbe)
    #    prédire sur X_train ET sur X_test (avec la MEME découpe de colonnes!)
    #    puis perf_train.append(accuracy_score(...)) et perf_test.append(accuracy_score(...))

    #  TODO 

plt.figure(facecolor='white')
plt.plot(all_ndim, perf_train, 'o-')
plt.plot(all_ndim, perf_test,  'o-')
plt.grid()
plt.xlabel("Nombre de dimensions utilisées (2 utiles + du bruit)")
plt.ylabel("Taux de bonne classification")
plt.legend(['Train', 'Test'])
plt.title("Le fléau de la dimensionnalité")
#plt.savefig("fig/curse.png", bbox_inches='tight')

**Ce qu'il faut lire sur cette figure**  TODO 

<a id="sec-b"></a>
## B. Sélectionner un sous-ensemble de variables

Les stratégies d'analyse des poids du classifieur linéaire et de *feature importance* sur les forêts
aléatoires sont utiles... mais elles ont été vues dans le TP précédent et ne sont pas reprises ici.

Nous partons donc directement des deux familles supervisées annoncées en introduction: le **filtre** (B.1),
qui note chaque variable isolément, et l'**enveloppe** (B.2), qui évalue des sous-ensembles à l'aide d'un
modèle. Gardez en tête la différence de coût: le filtre demande $d$ calculs élémentaires, l'enveloppe demande
des *centaines d'apprentissages*.

<a id="sec-b1"></a>
### B.1 Filtre naïf: corrélation entre chaque variable et $y$

L'idée la plus simple qui soit: une variable utile doit *varier avec* l'étiquette. On calcule donc, pour
chaque variable $j$, un score de liaison avec $y$, et on garde les meilleures.

Attention à un détail qui a son importance: le produit scalaire brut $|X_j^\top y|$ n'est **pas** une
corrélation. Avec $y \in \{0,1\}$, il ne somme que les exemples de la classe 1 et dépend de la moyenne de la
variable ; avec $y \in \{-1,+1\}$ il devient une vraie différence entre classes. La version propre est le
coefficient de Pearson, borné entre -1 et 1, donc directement comparable d'une variable à l'autre.

In [ ]:
# calcul de la liaison entre chaque variable et y, en trois versions de plus en plus propres

corr_brut = np.abs(X_train.T @ y_train)                         # produit scalaire brut, y dans {0,1}
corr_cent = np.abs(X_train.T @ (y_train*2-1)) / len(y_train)    # y recentré sur {-1,+1}
corr_pear = np.abs([np.corrcoef(X_train[:,j], y_train)[0,1]     # vraie corrélation de Pearson
                    for j in range(X_train.shape[1])])

plt.figure(facecolor='white', figsize=[13,3.5])
for k, (c, titre) in enumerate([(corr_brut, "produit scalaire brut"),
                                (corr_cent, "y recentré sur {-1,+1}"),
                                (corr_pear, "corrélation de Pearson")]):
    plt.subplot(1,3,k+1)
    plt.bar(np.arange(len(c)), c, color=['C3','C3'] + ['C0']*(len(c)-2))
    plt.title(titre)
    plt.xlabel("variable")
plt.tight_layout()
#plt.savefig("fig/corr_var.png")

print("Pearson  | variables utiles : {:.2f}, {:.2f}".format(corr_pear[0], corr_pear[1]))
print("         | maximum sur les 20 variables de bruit : {:.2f}".format(corr_pear[2:].max()))

# ... Sur un exemple jouet, ça marche vraiment très bien: les deux variables utiles (en rouge) sortent
# largement du lot, quelle que soit la version du critère.
# A tester sur des exemples réels, on ne sait jamais, le critère est intéressant !
# ATTENTION tout de même à ses deux angles morts, explorés en E.1 et E.2:
#   - il ne voit pas une variable utile UNIQUEMENT en interaction avec une autre (cf. le XOR, E.1)
#   - il ne voit pas la redondance: deux copies de la même variable ont deux fois le même bon score (E.2)

<a id="sec-b2"></a>
### B.2 Sélection séquentielle: une approche enveloppe

Élimination successive des dimensions les moins intéressantes (*backward*), ou ajout successif des dimensions
les plus intéressantes (*forward*).

* Avant toute chose: il faut un **critère** de sélection (ici, le taux de bonne classification d'un SVC)
* doc: [SequentialFeatureSelector](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html)
* Stratégie potentiellement **très couteuse**: il faut bien regarder les paramètres

Point de vue syntaxique, c'est la logique habituelle des transformations scikit-learn:

* `fit` fait tous les calculs (= trouve les variables à garder)
* `transform` applique la sélection sur `X` et `X_test` (= élimine effectivement les colonnes non retenues)

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector

estimator = svm.SVC(kernel="linear")    # je choisis d'utiliser le score en classif d'un SVM linéaire
selector  = SequentialFeatureSelector(estimator, n_features_to_select=2)  # le nombre de dimensions à garder
                                        # est spécifié... Pas très réaliste: en général, il faut le tester!
                                        # (cf. E.5 pour une version qui le choisit toute seule)
selector  = selector.fit(X_train, y_train)   # calcul des dimensions à conserver

print("variables retenues :", np.where(selector.get_support())[0])

# il est ensuite possible de filtrer les données:
Xnew = selector.transform(X_train)      # application du filtre
print("Ancienne dimension  : ", X_train.shape)
print("Nouvelle dimension  : ", Xnew.shape)

#### <span style="color:red">Mini-exo</span> forward contre backward

La cellule suivante chronomètre la version *forward* (celle par défaut).

1. Ajouter l'option qui bascule en *backward* et lancer le chronomètre
2. Bien comprendre l'algorithme qui tourne derrière, et prévoir **avant de lancer** lequel des deux sera le
   plus rapide ici. Combien d'apprentissages chaque version demande-t-elle pour passer de 22 à 2 variables?

In [ ]:
# impact en temps de calcul des options:
import time

# forward: on part de 0 variable et on en ajoute une à la fois, jusqu'à en avoir 2
t = time.time()
estimator = svm.SVC(kernel="linear")
selector  = SequentialFeatureSelector(estimator, n_features_to_select=2)
selector  = selector.fit(X_train, y_train)
print("forward  :", np.where(selector.get_support())[0], "en {:.2f}s".format(time.time()-t))

# backward
# Ajouter la bonne option et lancer le chronomètre.
# bien comprendre l'algorithme qui tourne derrière
# 4 lignes attendues: recopier le bloc forward ci-dessus en ajoutant UN SEUL argument:
#  1. t = time.time()
#  2. estimator = svm.SVC(kernel="linear")
#  3. selector = SequentialFeatureSelector(estimator, n_features_to_select=2, direction="backward")
#     puis selector.fit(X_train, y_train)
#  4. print des variables retenues et du temps écoulé
# AVANT DE LANCER: compter le nombre d'apprentissages nécessaires pour passer de 22 à 2 variables
# dans chaque sens (forward: on ajoute 1 variable à la fois en partant de 0 / backward: on en
# retire 1 à la fois en partant de 22) -- lequel va gagner?
###  TODO  ###

Réponse:

 TODO 

<a id="sec-c"></a>
## C. Construire de nouvelles variables: l'ACP

*Analyse en Composantes Principales* (PCA en anglais) —
[doc scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)

Changement complet de logique par rapport à la partie B: l'ACP ne **choisit** pas des variables parmi celles
qui existent, elle en **construit** de nouvelles, comme combinaisons linéaires des anciennes. Elle cherche les
directions qui *expliquent* le mieux les données, c'est-à-dire celles de plus grande variance.

Deux conséquences majeures, à garder en tête pendant toute cette partie:

* le critère est **non supervisé**: il n'utilise pas $y$ du tout. C'est une force (il s'applique aussi à des
  données non étiquetées, et il ne peut pas sur-apprendre les étiquettes) et une faiblesse (rien ne garantit
  que la direction la plus étalée soit la direction qui sépare les classes — c'est l'objet de [E.4](#sec-e4));
* on perd l'**interprétabilité**: "l'axe 1" est un mélange de toutes les variables d'origine, contrairement à
  une sélection qui, elle, désigne nommément des variables existantes.

L'analyse des valeurs propres permet de choisir combien d'axes conserver.

<a id="sec-c1"></a>
### C.1 Valeurs propres: combien d'axes conserver?

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()          # on peut préciser le nombre de valeurs propres à
                     # conserver / calculer: n_components=10
pca.fit(X_train)     # calcul des valeurs propres et des vecteurs propres (y n'est PAS utilisé)

# Analyse des valeurs propres:
plt.figure(facecolor='white', figsize=[11,4])
plt.subplot(1,2,1)
plt.bar(np.arange(X_train.shape[1]), pca.singular_values_)
plt.xlabel("axe")
plt.ylabel("valeur singulière")
plt.title("Valeurs singulières")

plt.subplot(1,2,2)
plt.plot(pca.explained_variance_ratio_, 'o-', label="par axe")
plt.plot(np.cumsum(pca.explained_variance_ratio_), 's-', label="cumulée")
plt.axhline(0.9, color='r', ls='--', lw=1, label="90% de variance")
plt.xlabel("axe")
plt.ylabel("part de variance expliquée")
plt.legend()
plt.grid()
plt.tight_layout()
#plt.savefig('fig/valp.png')

print("variance expliquée par l'axe 1 : {:.1%}".format(pca.explained_variance_ratio_[0]))
print("axes nécessaires pour 90% de la variance : {}".format(
    int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.9)) + 1))

<img src="fig/valp.png">

**Lecture de la figure**: un premier axe se détache nettement des autres, qui forment ensuite un plateau
décroissant très régulier. Ce plateau, c'est la signature du bruit — 20 directions statistiquement
équivalentes, sans structure. Le décrochage entre l'axe 1 et le reste (la "cassure" du diagramme, *elbow* ou
diagramme en éboulis / *scree plot*) est le critère graphique usuel: on garde les axes situés **avant** la
cassure.

Attention au piège du critère "90% de la variance", très populaire et ici très mauvais conseiller: comme le
bruit occupe 20 dimensions sur 22, il faut en garder une quinzaine pour atteindre 90%... alors qu'un seul axe
suffit, comme la suite va le montrer. **La variance n'est pas l'information.**

<a id="sec-c2"></a>
### C.2 Performances en fonction du nombre d'axes

Projetons les données sur les premiers axes, apprenons un classifieur et comparons les performances à
l'espace de représentation d'origine.

<img src="fig/pca.png">

À compléter: une boucle qui projette sur 1, 2, 3, ... $d$ vecteurs propres, apprend un modèle et évalue.

Questions:
1. Sur combien de dimensions faut-il projeter?
2. Comment expliquer que les performances sur **une seule** dimension soient quasi-optimales, alors que
   l'information de départ était répartie sur deux variables?
3. Pourquoi retrouve-t-on exactement le piètre niveau de performance de la partie A quand on garde tous
   les axes?

In [ ]:
# projection des données dans l'espace des composantes principales
# + apprentissage de modèle
# + évaluation des performances

Xnew  = pca.transform(X_train)   # projection sur tous les axes
XnewT = pca.transform(X_test)    # idem pour le test: ATTENTION, on réutilise la pca apprise sur le TRAIN

# boucle for:
#   Xd = projection sur 1, 2, 3, ... d vecteurs propres
#   Apprentissage modèle
#   Evaluation des performances
# => Conclusion: sur combien de dimensions faut-il projeter?

# C'est exactement la boucle de la partie A.3, avec Xnew à la place de X_train:
#  1. avant la boucle: mod = svm.SVC(gamma=0.1), 2 listes vides, all_ndim = range(1, 23)
#  2. dans la boucle: Xd = Xnew[:,:ndim] et penser à projeter les données de test
#     (les axes sont déjà triés par variance décroissante: prendre les ndim premières colonnes
#      revient donc bien à garder les ndim axes les plus importants)
#  3. mod.fit(Xd, y_train), puis accuracy_score en apprentissage et en test -> append
#  4. tracé des 2 courbes en fonction de all_ndim
#  5. Repérer le nombre d'axes optimal, à comparer aux 22 axes

###  TODO  ###

Réponse:

 TODO 

<a id="sec-c3"></a>
### C.3 Réduction en 2D: la visualisation [OPT, à garder pour la fin]

Une retombée très utile de l'ACP, indépendante de toute performance de classification: projeter en 1 ou 2
dimensions permet enfin de **regarder** des données qui vivaient en dimension 22.

In [ ]:
# affichage du nuage de points projeté en 1D et 2D

# 3 sous-figures (plt.subplot(1,3,k)):
#  1. les 2 dimensions d'ORIGINE (X_train[:,0], X_train[:,1]), sur lesquelles on superpose les
#     2 premiers axes de l'ACP: plt.plot de [0,0] vers pca.components_[0] et [1], mis à l'échelle
#     pour être visibles (le vecteur propre est de norme 1, donc minuscule sur ce nuage)
#  2. la projection 1D: plt.scatter(Xnew[:,0], [0]*len(X_train), c=y_train) -- 1 seule coordonnée
#  3. la projection 2D: plt.scatter(Xnew[:,0], Xnew[:,1], c=y_train)
# Puis afficher pca.components_[0][:2] et [1][:2]: sur quelle direction l'axe 1 s'est-il aligné?

###  TODO  ###

L'axe 1 (courbe bleue du premier graphe) est bien aligné sur la diagonale qui sépare les deux classes:
une seule coordonnée suffit donc à les distinguer, comme le montre le graphe du milieu. L'axe 2, lui, est
presque invisible dans le plan d'origine (ses deux premières coordonnées sont proches de 0): il pointe
essentiellement vers les dimensions de bruit, ce qui explique que le troisième graphe n'apporte rien de plus
que le deuxième.

<a id="sec-c4"></a>
### C.4 Application aux données USPS

Peut-on utiliser l'ACP sur les données USPS avec lesquelles nous avons joué il y a quelques semaines?
OUI! Évidemment. Et l'enjeu y est bien plus concret que sur les données jouets: 256 dimensions, dont beaucoup
de pixels de bord qui sont noirs sur *toutes* les images.

1. Visualiser les données USPS en 2 dimensions
2. Afficher les axes de projection eux-mêmes — ce sont des vecteurs de dimension 256, donc des **images**
3. Mesurer la performance en fonction du nombre d'axes conservés, et la comparer aux 256 dimensions d'origine

**Note**: les variables USPS sont préfixées par `u` (`Xu_train`, ...) pour ne pas écraser les données jouets
utilisées dans le reste du notebook.

In [ ]:
# Chargement des données
import pickle as pkl

data = pkl.load(open("data/usps.pkl", 'rb'))
# data est un dictionnaire contenant les champs explicites X_train, X_test, Y_train, Y_test
Xu_train = np.array(data["X_train"], dtype=float)   # changement de type pour éviter les problèmes d'affichage
Xu_test  = np.array(data["X_test"],  dtype=float)
Yu_train = data["Y_train"]
Yu_test  = data["Y_test"]

# pour rappel sur la structuration des données: affichage de l'image 18 avec reshape
plt.figure(facecolor='white', figsize=[3,3])
plt.imshow(Xu_train[18].reshape(16,16), cmap="gray")
plt.title("Image de : {}".format(Yu_train[18]))

print("USPS: {} exemples d'apprentissage, {} de test, {} pixels".format(
    Xu_train.shape[0], Xu_test.shape[0], Xu_train.shape[1]))

In [ ]:
# 1. Faire la projection
#    pca_u = PCA(n_components=50).fit(Xu_train), puis .transform() sur le TRAIN et sur le TEST
#    (ATTENTION: une seule ACP, apprise sur le train, appliquée aux deux jeux)
# 2. Affichage de la base projetée sur les deux premiers axes
#    plt.scatter(Zu[:,0], Zu[:,1], c=Yu_train, cmap='tab10', s=4) + colorbar
#    + en 2e subplot: la variance expliquée cumulée, np.cumsum(pca_u.explained_variance_ratio_)
# 3. Afficher également les axes de projection eux-mêmes
#    chaque pca_u.components_[k] est un vecteur de 256 valeurs = une image 16x16!
#    => boucle d'imshow(pca_u.components_[k].reshape(16,16)) sur les 8 premiers axes
#    (+ l'image moyenne Xu_train.mean(0) en guise de comparaison)
# 4. Calculer la performance en fonction du nombre d'axes utilisés
#    boucle sur nd in [2, 5, 10, 20, 50]: svm.SVC().fit(Zu[:,:nd], Yu_train) puis accuracy en test
#    + la référence obtenue sur les 256 dimensions d'origine

###  TODO  ###

**Bilan sur USPS**  TODO 

#### <span style="color:red">Mini-exo</span> faut-il normaliser les pixels avant l'ACP?

L'ACP maximise la **variance**: une variable exprimée dans une grande unité écrase toutes les autres. La
question de normaliser (`StandardScaler`) avant de projeter se pose donc systématiquement... sauf quand toutes
les variables sont déjà dans la même unité, ce qui est le cas ici (des intensités de pixels).

Regardons tout de même la distribution de quelques pixels avant de conclure.

In [ ]:
# 2 choses à produire:
#  1. les histogrammes (plt.hist) de quelques pixels bien choisis: un pixel de bord (0), un pixel
#     du centre (128)... en affichant leur variance dans le titre
#  2. v = Xu_train.var(0), puis comparer min / médiane / max: quel est le rapport max/min?
#     compter également les pixels qui valent 0 sur plus de 90% des images
#     (np.mean(Xu_train == 0, axis=0) > 0.9)
# Puis CONCLURE: que ferait StandardScaler à un pixel de bord de variance quasi nulle?

###  TODO  ###

<a id="sec-d"></a>
## D. Régularisation: sélectionner pendant l'apprentissage

La promesse: sélectionner les bonnes dimensions **au cours** de l'apprentissage, sans étape séparée. Nous
avons vu les mécanismes dans la séance sur le gradient.

* Pénalisation extrême des poids $\Rightarrow$ aucune dimension retenue (= simplicité extrême)... mais
  performances très faibles.
* Pénalisation faible $\Rightarrow$ retour à la situation initiale (bonnes performances en apprentissage,
  mais risque de sur-apprentissage)
* **Question**: quid des situations intermédiaires? Quel compromis entre parcimonie et performances?

Nous allons envisager plusieurs formulations, faire varier la régularisation et étudier à la fois la
performance et la parcimonie. Comme indicateur de parcimonie, nous étudierons la proportion de $w$ non nuls:

$$Parcimonie = \frac{\sum_j I_{w_j \ne 0}}{d}$$

**Convention de lecture des trois figures qui suivent**: l'axe des abscisses est l'indice dans la liste des
régularisations testées, ordonnée de la **plus forte à la plus faible**. On lit donc les courbes de gauche
(modèle contraint et simple) vers la droite (modèle libre et complexe).

In [ ]:
# (re)génération des données bruitées
# indispensable ici: les variables X_train / X_test de la partie A ont pu être modifiées entre temps,
# et on veut que cette partie D soit exécutable indépendamment du reste.

np.random.seed(0)

centers      = [[-2.0, -2.0], [2.0, 2.0]]
clusters_std = [1.5, 1.5]
X, y = make_blobs(n_samples=100, centers=centers, cluster_std=clusters_std,
                  n_features=2, random_state=0)

ndim_noise = 20
Noise      = np.random.randn(len(X), ndim_noise) * clusters_std[0]
Xn         = np.concatenate((X, Noise), axis=1)

Xn_train, Xn_test, yn_train, yn_test = train_test_split(Xn, y, test_size=0.33, random_state=0)

# la parcimonie idéale que l'on aimerait atteindre: 2 variables sur 22
print("parcimonie visée : {}/{} = {:.2f}".format(2, Xn.shape[1], 2/Xn.shape[1]))

<a id="sec-d1"></a>
### D.1 Régularisation L2 (Ridge)

Nous avons déjà croisé la régularisation L2 avec les SVM. Essayons de pousser un peu plus loin l'analyse de
ce type d'approches:

$$\text{Formulation Ridge: }\qquad \mathcal L = \sum_{i=1}^n \left( \sum_{j=1}^d w_j x_{ij} - y_i\right)^2 + \textcolor{red}{C} \|w\|^2$$

Après la phase de génération de données, nous allons faire varier $C$ et étudier les performances par rapport
au nombre de variables retenues.

À compléter: l'évaluation et le comptage des coefficients non nuls.

In [ ]:
from sklearn.linear_model import RidgeClassifier

# différentes valeurs de C, de la plus forte régularisation à la plus faible
all_a = 10**(np.linspace(10,-10,11))
p_a = []  # perf en apprentissage
p_t = []  # perf en test
wc  = []  # proportion de coefficients non nuls

for a in all_a:  # boucle de test de régularisation
    mod = RidgeClassifier(alpha=a)
    mod.fit(Xn_train, yn_train)
    # Compléter l'évaluation et le comptage des coefficients non nuls
    # 3 lignes attendues (mod vient d'être appris juste au-dessus):
    #  1. prédire sur Xn_train et sur Xn_test
    #  2. p_a.append(accuracy_score(...)) et p_t.append(accuracy_score(...))
    #  3. wc.append(proportion de coefficients non nuls dans mod.coef_[0])
    #     en L2 les poids ne sont JAMAIS exactement nuls, il faut donc SEUILLER pour les compter:
    #     np.where(np.abs(mod.coef_[0]) > 1e-5, 1, 0).mean()
    #     -- le fait même de devoir seuiller est déjà un indice sur le résultat à venir...
    ###   TODO  ###

plt.figure(facecolor='white', figsize=(10,5))
plt.plot(p_a, 'o-')
plt.plot(p_t, 'o-')
plt.plot(wc,  's-')
plt.grid()
plt.xlabel('Régularisation (décroissante)')
plt.ylabel('Perf / Proportion de dimensions')
plt.legend(['P train', 'P test', '#w'])
plt.title("Ridge (L2): la performance varie, la parcimonie non")
# plt.savefig('fig/reg_L2.pdf')

<a id="sec-d2"></a>
### D.2 Régularisation L1 (LASSO)

Introduction d'une régularisation parcimonieuse:

$$\text{Formulation LASSO: }\qquad \mathcal L = \sum_{i=1}^n \left( \sum_{j=1}^d w_j x_{ij} - y_i\right)^2 + \textcolor{red}{C} \sum_{j=1}^d |w_j|$$

Après la phase de génération de données, nous allons faire varier $C$ et étudier performances contre
parcimonie.

**Pourquoi la L1 annule-t-elle des poids alors que la L2 ne le fait pas?** Parce que la dérivée de $|w_j|$
vaut $\pm 1$ même au voisinage de zéro: la pénalité continue de "pousser" le poids vers 0 avec la même force,
et l'y maintient exactement. La dérivée de $w_j^2$ vaut $2w_j$, qui s'évanouit quand $w_j$ approche 0: la
pénalité L2 écrase les poids mais ne les annule jamais.

In [ ]:
from sklearn.linear_model import lasso_path

# PIEGE A CONNAITRE: contrairement à Lasso() ou ElasticNet(), lasso_path ne gère PAS de biais
# (pas d'intercept). Sans précaution, la règle de décision "X.w > 0.5" est donc mal placée et les
# performances plafonnent à 0.70 alors que le modèle est bon. La parade standard: centrer X et y avant
# l'appel, et rajouter la moyenne de y à la prédiction.
X_moy = Xn_train.mean(axis=0)
y_moy = yn_train.mean()

# différentes valeurs de C sur les données d'origine (le balayage est fait automatiquement ici)
alphas, coefs, dual_gaps = lasso_path(Xn_train - X_moy, yn_train - y_moy)   # fit automatique à l'intérieur
# note: le modèle LASSO s'utilise comme les autres modèles (Lasso().fit(...)),
# pour gagner du temps j'ai juste calculé tous les modèles d'un coup :)
# coefs a pour dimensions (nb de variables, nb de valeurs de alpha), alphas est DECROISSANT

p_a = []
p_t = []
wc  = []

for i in range(coefs.shape[1]):

    yhata = np.where((Xn_train - X_moy).dot(coefs[:,i]) + y_moy > 0.5, 1, 0)
    yhatt = np.where((Xn_test  - X_moy).dot(coefs[:,i]) + y_moy > 0.5, 1, 0)
    # ATTENTION, c'est un modèle de régression: il faut fabriquer les étiquettes à la main
    p_a.append(accuracy_score(yn_train, yhata))
    p_t.append(accuracy_score(yn_test,  yhatt))
    wc.append(np.where(np.abs(coefs[:,i]) > 1e-5, 1, 0).mean())

plt.figure(facecolor='white', figsize=(10,5))
plt.plot(p_a, 'o-')
plt.plot(p_t, 'o-')
plt.plot(wc,  's-')
plt.grid()
plt.xlabel('Régularisation (décroissante)')
plt.ylabel('Perf / Proportion de dimensions')
plt.legend(['P train', 'P test', '#w'])
plt.title("LASSO (L1): la parcimonie s'installe toute seule")
# plt.savefig('fig/reg_L1.pdf')

# le meilleur compromis: la performance maximale avec le moins de variables possible
i_best = int(np.argmax(p_t))
print("meilleur test : {:.3f} avec {:.0f} variables sur {} (alpha={:.3f})".format(
    p_t[i_best], wc[i_best]*Xn_train.shape[1], Xn_train.shape[1], alphas[i_best]))
print("variables retenues :", np.where(np.abs(coefs[:,i_best]) > 1e-5)[0])

<a id="sec-d3"></a>
### D.3 Régularisation Elastic Net

Combinaison des deux régularisations L2 et L1. L'idée est de jouer principalement sur la L2 et de mettre un
peu de L1 pour la parcimonie:

$$\mathcal L = \sum_{i=1}^n \left( \sum_{j=1}^d w_j x_{ij} - y_i\right)^2 + \textcolor{red}{C} \left( \rho \sum_{j=1}^d |w_j| + \frac{1-\rho}{2}\|w\|^2 \right)$$

Le paramètre $\rho$ est le `l1_ratio` de scikit-learn:
[doc ElasticNet](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html).

À compléter: l'évaluation et le comptage des coefficients non nuls.

In [ ]:
from sklearn.linear_model import ElasticNet

# différentes valeurs de C, de la plus forte à la plus faible
all_a = (np.linspace(2.5, 0.1, 20))
p_a = []  # perf en apprentissage
p_t = []  # perf en test
wc  = []  # proportion de coefficients non nuls

for a in all_a:
    mod = ElasticNet(alpha=a, l1_ratio=0.5)
    mod.fit(Xn_train, yn_train)
    # Compléter l'évaluation et le comptage des coefficients non nuls
    # ATTENTION: par défaut, il s'agit d'un modèle de régression:
    # il faut remettre les classes en face des prédictions pour utiliser les métriques
    # 3 lignes attendues (calquées sur la boucle Ridge de D.1):
    #  1. yhata = np.where(mod.predict(Xn_train) > 0.5, 1, 0)   <- le seuillage en classes {0,1}
    #     idem pour yhatt sur Xn_test
    #  2. p_a.append(...) et p_t.append(...) avec accuracy_score
    #  3. wc.append(...) sur mod.coef_ -- ATTENTION: PAS de [0] ici! ElasticNet étant un modèle de
    #     régression, coef_ est un simple vecteur (contrairement au RidgeClassifier de D.1)
    ###   TODO  ###

plt.figure(facecolor='white', figsize=(10,5))
plt.plot(p_a, 'o-')
plt.plot(p_t, 'o-')
plt.plot(wc, '+-')
plt.grid()
plt.xlabel('Régularisation (décroissante)')
plt.ylabel('Perf / Proportion de dimensions')
plt.legend(['P train', 'P test', '#w'])
plt.title("Elastic Net: le compromis entre les deux")
#plt.savefig('fig/reg_L1L2.pdf')

<a id="sec-d4"></a>
### D.4 Synthèse des trois régularisations

**Ce que montrent les trois figures**  TODO 

<a id="sec-e"></a>
## E. Exercices complémentaires

Les exercices de cette partie sont indépendants les uns des autres (chacun régénère ses propres données) et
peuvent être traités dans le désordre, selon le temps disponible. Ils prolongent chacun un point précis des
parties A à D.

| Exercice | Notion travaillée | Prolonge |
|---|---|---|
| E.1 | filtres univariés de scikit-learn, et leur angle mort: le XOR | B.1 |
| E.2 | variables redondantes: pourquoi un filtre se fait piéger et pas une enveloppe | B.1, B.2 |
| E.3 | sélectionner avant la validation croisée: une fuite de données spectaculaire | B |
| E.4 | deux pièges de l'ACP: l'échelle des variables, et l'absence d'étiquettes (LDA) | C |
| E.5 | élimination récursive (RFE) et choix automatique du nombre de variables (RFECV) | B.2 |
| E.6 | stabilité de la sélection: LASSO contre Elastic Net sur variables corrélées | D |

<a id="sec-e1"></a>
### E.1 Filtres univariés et leur angle mort

La corrélation calculée "à la main" en B.1 est un **filtre univarié**. Scikit-learn en propose une version
industrialisée, avec plusieurs scores au choix:

* `f_classif`: test statistique ANOVA (rapport de la variance inter-classes sur la variance intra-classe),
  il ne détecte que les liaisons **linéaires** ou monotones;
* `mutual_info_classif`: information mutuelle, capable en principe de détecter des liaisons non linéaires;
* `SelectKBest(score, k)` garde les `k` meilleures variables, `SelectPercentile` un pourcentage.

L'exercice a deux temps: vérifier que ça marche, puis trouver le cas où ça casse.

1. Sur les données bruitées de la partie A, appliquer `SelectKBest(f_classif, k=2)` et vérifier que les
   variables 0 et 1 sont bien retrouvées. Regarder aussi les p-values (`selector.pvalues_`)
2. Construire maintenant un problème **XOR** en 2 dimensions: `X = rng.randn(400,2)` et
   `y = (X[:,0] > 0) ^ (X[:,1] > 0)`, auquel on ajoute 5 dimensions de bruit
3. Sur ce problème, calculer les scores `f_classif` **et** `mutual_info_classif`, puis appliquer
   `SelectKBest(f_classif, k=2)`: quelles variables sont retenues?
4. Mesurer par validation croisée la performance d'un `SVC` à noyau gaussien sur les 2 vraies variables,
   puis appliquer une sélection séquentielle (partie B.2) sur les 7 variables
5. Répondre:
    * pourquoi le filtre univarié échoue-t-il complètement sur le XOR?
    * l'information mutuelle s'en sort-elle mieux? Pourquoi?
    * qu'est-ce que cela dit du bon usage des filtres dans une chaine de traitement?

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import cross_val_score

# le problème XOR: chaque variable prise SEULE est indépendante de y
rng = np.random.RandomState(0)
Xx  = rng.randn(400,2)
yx  = ((Xx[:,0] > 0) ^ (Xx[:,1] > 0)).astype(int)
Xx  = np.concatenate((Xx, rng.randn(400,5)), axis=1)   # + 5 dimensions de bruit

plt.figure(facecolor='white', figsize=[4,4])
plt.scatter(Xx[:,0], Xx[:,1], c=yx, s=10)
plt.xlabel("variable 0"); plt.ylabel("variable 1")
plt.title("XOR: séparable, mais pas variable par variable")

# Schéma de résolution (numérotation de l'énoncé, ~12 lignes):
#  1. sur les données de la partie A: sel = SelectKBest(f_classif, k=2).fit(X_train, y_train)
#     afficher np.where(sel.get_support())[0], puis sel.scores_ et sel.pvalues_
#     => le filtre retrouve-t-il bien les variables 0 et 1? avec quelle marge sur le bruit?
#  3. sur le XOR: F, p = f_classif(Xx, yx) et mi = mutual_info_classif(Xx, yx, random_state=0)
#     afficher les 7 scores de chaque critère, puis regarder ce que retient
#     SelectKBest(f_classif, k=2).fit(Xx, yx)
#  4. cross_val_score(svm.SVC(), Xx[:,:2], yx, cv=5).mean() => l'information EST pourtant bien là
#     puis SequentialFeatureSelector(svm.SVC(), n_features_to_select=2, cv=5).fit(Xx, yx)
#     => une approche ENVELOPPE retrouve-t-elle, elle, les bonnes variables?

###  TODO  ###

Réponse:

 TODO 

<a id="sec-e2"></a>
### E.2 Variables redondantes: filtre contre enveloppe

Deuxième angle mort du filtre univarié, encore plus fréquent en pratique que le XOR: la **redondance**. Sur
des données réelles, il est très courant que plusieurs variables mesurent à peu près la même chose (deux
capteurs voisins, une taille en cm et en pouces, un chiffre d'affaires et un chiffre d'affaires hors taxes).

1. Construire un jeu à 12 variables: la variable utile `v0`, **5 copies bruitées** de `v0`, la seconde
   variable utile `v1`, et 5 variables de bruit
2. Calculer les scores `f_classif`: que valent les scores des 6 variables du groupe redondant?
3. Appliquer `SelectKBest(k=2)` puis une sélection séquentielle *forward* avec 2 variables. Comparer les
   deux sous-ensembles obtenus
4. Mesurer la performance en test d'un SVM linéaire sur chacun des deux sous-ensembles
5. Répondre:
    * quelle erreur exacte commet le filtre, et pourquoi ne peut-il structurellement pas l'éviter?
    * l'enveloppe fait-elle mieux? À quel prix?

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, SequentialFeatureSelector

# 1. le jeu à variables redondantes: v0, 5 quasi-copies de v0, v1, puis 5 variables de bruit
rng = np.random.RandomState(0)
Xb, yb = make_blobs(n_samples=200, centers=[[-2.,-2.],[2.,2.]], cluster_std=[1.5,1.5],
                    n_features=2, random_state=0)

copies = Xb[:,[0]] + 0.1*rng.randn(len(Xb), 5)        # 5 quasi-copies de la variable 0
Xr     = np.concatenate((Xb[:,[0]], copies, Xb[:,[1]], rng.randn(len(Xb), 5)), axis=1)
noms   = ['v0'] + ['copie{}'.format(i) for i in range(1,6)] + ['v1'] + ['bruit{}'.format(i) for i in range(5)]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yb, test_size=0.33, random_state=0)
print("variables :", noms)

# Schéma de résolution (numérotation de l'énoncé, ~10 lignes):
#  2. F, p = f_classif(Xr_train, yr_train), puis afficher le score des 12 variables
#     (boucle sur zip(noms, F)) => que valent les scores des 6 variables du groupe redondant?
#  3. i_filtre  = les 2 variables retenues par SelectKBest(f_classif, k=2)
#     i_wrapper = les 2 variables retenues par SequentialFeatureSelector(SVC(kernel='linear'),
#                 n_features_to_select=2)
#     np.where(....get_support())[0] dans les deux cas, à afficher en NOMS: [noms[i] for i in idx]
#  4. pour chacun des 2 sous-ensembles (+ la vérité terrain [0,6]): apprendre un SVM linéaire sur
#     Xr_train[:,idx] et mesurer l'accuracy sur Xr_test[:,idx]

###  TODO  ###

Réponse:

 TODO 

<a id="sec-e3"></a>
### E.3 Sélectionner avant la validation croisée: une fuite de données

Voici l'erreur la plus fréquente — et la plus grave — de toute la chaine de sélection de variables. Elle est
tellement classique qu'elle a un nom: la **fuite de données** (*data leakage*), et elle a produit des
publications entières de résultats faux, notamment en bio-informatique.

Le scénario est parfaitement innocent en apparence: (1) je sélectionne les 20 meilleures variables sur mon jeu
de données, (2) je mesure honnêtement la performance de mon modèle par validation croisée sur ces 20
variables. Le piège: à l'étape 1, j'ai déjà **regardé toutes les étiquettes**, y compris celles des plis qui
serviront de test à l'étape 2.

Pour que la démonstration soit sans appel, on va travailler sur des données qui ne contiennent, par
construction, **strictement aucune information**: du bruit gaussien, et des étiquettes tirées à pile ou face.
Toute performance supérieure à 0.5 sera donc, à coup sûr, un artefact.

1. Générer `X = rng.randn(100, 5000)` et `y = rng.randint(0, 2, 100)`
2. **Mauvaise pratique**: appliquer `SelectKBest(f_classif, k=20)` sur *tout* `X`, puis mesurer
   `cross_val_score` d'un SVM linéaire sur les 20 variables retenues
3. **Bonne pratique**: mettre la sélection et le classifieur dans un `Pipeline`, et passer ce pipeline entier
   à `cross_val_score`
4. Répéter sur 10 jeux de données pour montrer que l'écart n'est pas un accident
5. Répondre:
    * quelle performance obtient chaque protocole? Laquelle est la bonne réponse attendue?
    * expliquer précisément *où* fuient les étiquettes dans le premier protocole
    * pourquoi le phénomène est-il d'autant plus violent que $d$ est grand par rapport à $n$?
    * quelles autres étapes d'une chaine de traitement doivent, pour la même raison, être placées dans le
      pipeline?

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score

# Schéma de résolution (~15 lignes):
#  1. boucler sur 10 jeux de données (rs in range(10)):
#     Xf = rng.randn(100, 5000) et yf = rng.randint(0, 2, 100)  => QUE du bruit, aucune information
#  2. MAUVAISE pratique: Xsel = SelectKBest(f_classif, k=20).fit_transform(Xf, yf) -- sur TOUT X --
#     puis cross_val_score(svm.SVC(kernel='linear'), Xsel, yf, cv=5).mean()
#  3. BONNE pratique: pipe = Pipeline([('sel', SelectKBest(f_classif, k=20)),
#                                      ('clf', svm.SVC(kernel='linear'))])
#     puis cross_val_score(pipe, Xf, yf, cv=5).mean()
#     => la sélection est alors refaite DANS chaque pli, sur le seul jeu d'apprentissage du pli
#  4. afficher les 2 moyennes (+/- écart-type) et les comparer à 0.500, la seule réponse acceptable
#  5. tracer les 2 séries de 10 valeurs, avec une horizontale à 0.5 (plt.axhline)

###  TODO  ###

Réponse:

 TODO 

<a id="sec-e4"></a>
### E.4 Deux pièges de l'ACP: l'échelle et l'absence d'étiquettes

L'ACP maximise la **variance**. Cette phrase, prise au sérieux, contient les deux pièges de la méthode.

**Piège 1 — la variance dépend de l'unité de mesure.** Une distance exprimée en millimètres a une variance
un million de fois plus grande que la même distance en mètres. Une variable inutile mais "en grands nombres"
peut donc capter le premier axe à elle seule.

**Piège 2 — la variance n'est pas la discrimination.** L'ACP ignore $y$: rien ne garantit que la direction la
plus étalée soit celle qui sépare les classes. L'alternative supervisée est l'**analyse discriminante
linéaire** (`LinearDiscriminantAnalysis`), qui cherche les directions maximisant le rapport de la variance
inter-classes sur la variance intra-classes.

**Partie 1 — l'échelle**
1. Reprendre les données bruitées de la partie A, et multiplier **une seule variable de bruit** par 50
   (par exemple `Xn[:,5] *= 50`)
2. Comparer deux pipelines: `PCA(2) + SVC` et `StandardScaler + PCA(2) + SVC`
3. Regarder sur quelle variable l'axe 1 met le plus de poids dans chaque cas (`np.argmax(np.abs(pca.components_[0]))`)

**Partie 2 — l'absence d'étiquettes**
4. Construire un nuage très allongé selon $x_1$ (écart-type 10) dont les deux classes ne diffèrent que
   selon $x_2$ (écart-type 0.5, décalage de 2 entre classes)
5. Afficher le nuage, puis comparer `PCA(n_components=1) + SVC` et `LDA(n_components=1) + SVC`
6. Répondre:
    * quelle part de variance le premier axe de l'ACP capture-t-il, et quelle performance obtient-on?
    * quand faut-il préférer la LDA à l'ACP, et quelle est sa limite (indice: combien d'axes une LDA
      peut-elle produire avec $K$ classes)?

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

# ---------- Partie 1: l'échelle des variables ----------
rng = np.random.RandomState(0)
Xe, ye = make_blobs(n_samples=200, centers=[[-2.,-2.],[2.,2.]], cluster_std=[1.5,1.5],
                    n_features=2, random_state=0)
Xe = np.concatenate((Xe, rng.randn(len(Xe), 20)*1.5), axis=1)
Xe[:,5] *= 50          # une variable de bruit exprimée dans une unité 50x plus grande
Xe_train, Xe_test, ye_train, ye_test = train_test_split(Xe, ye, test_size=0.33, random_state=0)

# ---------- Partie 2: un nuage allongé dont les classes ne diffèrent que selon x2 ----------
rng = np.random.RandomState(0)
n   = 300
lab = rng.randint(0, 2, n)
Xl  = np.c_[rng.randn(n)*10,                  # direction très étalée... et inutile
            rng.randn(n)*0.5 + 2.0*lab]       # direction peu étalée... et discriminante
Xl_train, Xl_test, yl_train, yl_test = train_test_split(Xl, lab, test_size=0.33, random_state=0)

# Schéma de résolution (numérotation de l'énoncé, ~15 lignes):
# --- Partie 1: l'échelle des variables (données Xe) ---
#  2. comparer 2 pipelines: Pipeline([('pca', PCA(n_components=2)), ('clf', svm.SVC())])
#     et le même avec ('sc', StandardScaler()) placé EN PREMIER.
#     fit sur (Xe_train, ye_train), puis accuracy_score sur le test
#  3. apprendre les 2 ACP séparément (sur Xe_train brut / sur StandardScaler().fit_transform(...))
#     et regarder np.argmax(np.abs(pca.components_[0])): sur quelle variable l'axe 1 se pose-t-il
#     dans chaque cas? (rappel: c'est la variable 5 qui a été multipliée par 50)
# --- Partie 2: l'ACP ne connait pas y (données Xl) ---
#  5. pc = PCA(n_components=1).fit(Xl_train)
#     ld = LinearDiscriminantAnalysis().fit(Xl_train, yl_train)   <- lui reçoit les étiquettes
#     afficher les 2 directions (pc.components_[0], et ld.coef_[0] normalisé): sont-elles alignées?
#     tracer le nuage + une flèche (plt.arrow) matérialisant l'axe de l'ACP
#     puis comparer 3 pipelines: PCA(1)+SVC, LDA(1)+SVC, et SVC seul sur les 2 dims (référence)

###  TODO  ###

Réponse:

 TODO 

<a id="sec-e5"></a>
### E.5 Élimination récursive (RFE) et choix automatique du nombre de variables

La sélection séquentielle de B.2 réapprend un modèle pour *chaque variable candidate*, à *chaque* tour. La
**RFE** (*Recursive Feature Elimination*) exploite une information que l'on a déjà sous la main: les poids du
modèle. À chaque tour, elle apprend **un seul** modèle, supprime la ou les variables de plus petit poids, et
recommence. C'est beaucoup moins cher, au prix d'une hypothèse: le modèle doit exposer un `coef_` ou un
`feature_importances_`.

Reste une question laissée en suspens depuis B.2: **combien** de variables garder? `RFECV` y répond en
mesurant, par validation croisée, la performance à chaque taille de sous-ensemble.

1. Sur les données de la partie A, appliquer `RFE(SVC(kernel='linear'), n_features_to_select=2)` et
   chronométrer. Comparer au temps d'une sélection séquentielle *backward*
2. Regarder `rfe.ranking_`: que contient ce tableau?
3. Appliquer `RFECV(SVC(kernel='linear'), cv=5)` et tracer `cv_results_['mean_test_score']` en fonction du
   nombre de variables
4. Répondre:
    * quel gain de temps la RFE apporte-t-elle, et que suppose-t-elle en échange?
    * combien de variables `RFECV` retient-il? Est-ce le résultat attendu, et que faut-il en penser?
    * pourquoi un SVM **linéaire** est-il ici beaucoup moins sensible aux dimensions de bruit que le SVM à
      noyau gaussien de la partie A?

In [ ]:
from sklearn.feature_selection import RFE, RFECV, SequentialFeatureSelector
import time

# Schéma de résolution (~12 lignes):
#  1. chronométrer (time.time() avant / après) RFE(svm.SVC(kernel='linear'),
#     n_features_to_select=2).fit(X_train, y_train), puis la même chose avec
#     SequentialFeatureSelector(..., direction='backward'), et comparer les 2 temps
#     np.where(selecteur.get_support())[0] donne les variables retenues dans les deux cas
#  2. afficher rfe.ranking_: un tableau de 22 entiers. Que vaut-il pour les variables retenues,
#     et que signifie l'ordre des autres valeurs?
#  3. rfecv = RFECV(svm.SVC(kernel='linear'), cv=5, scoring='accuracy').fit(X_train, y_train)
#     rfecv.n_features_ = le nombre de variables choisi AUTOMATIQUEMENT
#     tracer rfecv.cv_results_['mean_test_score'] en fonction du nombre de variables
#     (plt.errorbar avec yerr=rfecv.cv_results_['std_test_score'] pour voir la dispersion),
#     + une verticale sur le choix de RFECV et une sur la vérité terrain (2 variables)

###  TODO  ###

Réponse:

 TODO 

<a id="sec-e6"></a>
### E.6 Stabilité de la sélection: LASSO contre Elastic Net

La partie D a laissé une question ouverte: à quoi sert vraiment l'Elastic Net, puisque le LASSO sélectionne
déjà? La réponse tient en un mot: la **stabilité**.

Quand plusieurs variables portent la même information (le cas de E.2), le LASSO n'a aucune raison de les
préférer l'une à l'autre: il en choisit **une, arbitrairement**, et annule les autres — c'est le prix de la
parcimonie. Changez très légèrement les données, et il choisira une autre variable du groupe. Pour un
scientifique qui veut *interpréter* les variables retenues ("quels gènes sont impliqués?"), c'est
catastrophique. Le terme L2 de l'Elastic Net corrige ce défaut: il pousse les variables corrélées à recevoir
des poids **semblables**, donc à être retenues ou éliminées ensemble (*grouping effect*).

Mesurons cela par **bootstrap**: on rééchantillonne les données avec remise, on refait la sélection, et on
compte la fréquence à laquelle chaque variable est retenue. C'est le principe de la *stability selection*.

1. Reprendre le jeu à variables redondantes de [E.2](#sec-e2) (`v0`, 5 copies, `v1`, 5 bruits)
2. Sur 100 rééchantillonnages bootstrap, ajuster un `Lasso(alpha=0.05)` et compter, pour chaque variable,
   la fréquence de sélection
3. Faire de même avec un `ElasticNet`. **Attention à la comparaison**: pour être équitable, il faut la même
   force de pénalité L1, c'est-à-dire `alpha_EN * l1_ratio = alpha_LASSO`
4. Tracer les deux profils de fréquence côte à côte
5. Répondre:
    * comment se comportent les 6 variables du groupe redondant dans chaque cas?
    * les deux méthodes rejettent-elles aussi bien le bruit?
    * laquelle utiliser pour *interpréter*, laquelle pour *prédire avec peu de variables*?

In [ ]:
from sklearn.linear_model import Lasso, ElasticNet

# 1. le jeu à variables redondantes (identique à E.2)
rng = np.random.RandomState(0)
Xb, yb = make_blobs(n_samples=200, centers=[[-2.,-2.],[2.,2.]], cluster_std=[1.5,1.5],
                    n_features=2, random_state=0)
copies = Xb[:,[0]] + 0.1*rng.randn(len(Xb), 5)
Xr     = np.concatenate((Xb[:,[0]], copies, Xb[:,[1]], rng.randn(len(Xb), 5)), axis=1)
noms   = ['v0'] + ['copie{}'.format(i) for i in range(1,6)] + ['v1'] + ['bruit{}'.format(i) for i in range(5)]

# Schéma de résolution (~15 lignes):
#  1. définir ALPHA_L1 = 0.05 et B = 100 (le nombre de rééchantillonnages), puis les 2 modèles:
#     Lasso(alpha=ALPHA_L1, max_iter=10000)
#     ElasticNet(alpha=ALPHA_L1/0.5, l1_ratio=0.5, max_iter=10000)
#     ATTENTION à l'équité de la comparaison: c'est le produit alpha*l1_ratio qui fixe la force de
#     la pénalité L1, d'où la division par 0.5
#  2. pour chaque modèle, boucler B fois:
#       idx = rng_b.choice(len(Xr), len(Xr), replace=True)    <- tirage AVEC remise (bootstrap)
#       mod.fit(Xr[idx], yb[idx]), puis cumuler (np.abs(mod.coef_) > 1e-8) dans un compteur
#     diviser le compteur par B => la fréquence de sélection de chaque variable
#     (repartir du MEME RandomState pour les 2 modèles: on veut comparer à tirages identiques)
#  3. afficher le tableau des 12 fréquences pour les 2 modèles, côte à côte
#  4. diagramme en barres groupées (plt.bar avec x-0.2 et x+0.2) + surlignage du groupe redondant
#     (plt.axvspan) pour comparer les 2 profils d'un coup d'oeil

###  TODO  ###

Réponse:

 TODO 

----
<a id="sec-annexe"></a>
## Annexe: transformation du notebook en version étudiante

La cellule ci-dessous produit `3-notebook-selfeat.ipynb` à partir de ce fichier: tout ce qui se trouve entre
les balises de correction (dans les cellules de code **comme** dans les cellules de texte) est remplacé par
un `TODO`.

**À faire avant de la lancer**: `Kernel > Restart & Clear Output`, sinon les sorties des cellules de
correction (résultats et figures) restent visibles dans la version distribuée.

In [ ]:
###  TODO  ###